# Multimodal captioning with a vision LLM

Generates captions for recipe images and user-review images with a local vision model (`gemma3` via Ollama) and merges them into the recipe and review data. Output: `augmented_food_recipe.json` and `data/augmented_user_review.json`.

> Part of **Connoisseur Companion**. Adapted from the IBM *RAG and Agentic AI* course on Coursera, reworked to run fully locally with Ollama.

In [ ]:
# Run from the repo: notebooks read and write files in `data/`
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent / "data")
print("Working directory:", Path.cwd())


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import os
from PIL import Image


# Libraries and codes to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

In [ ]:
import zipfile

with zipfile.ZipFile("synthetic-recipe-images.zip", 'r') as zip_ref:
    zip_ref.extractall()

## Preprocess the food recipe data

In [ ]:
### Step 1.1: Load the json file. Define the loaded data as recipe_data.
recipe_data = json.load(open('Recipes.json', encoding='utf-8'))

### Step 1.2: Print each key-value pair of the first recipe. In this format: key (type of value): value
for key, value in recipe_data[0].items():
    print(f"{key} ({type(value).__name__}): {value}")

### Step 1.3: Show the image of the first recipe (recipe1)
recipe1 = recipe_data[0]
Image.open(os.path.join("synthetic_recipe_images", f"recipe{recipe1['id']}.png"))

In [ ]:
import base64
import ollama

def vision_llm(system_msg, prompt_txt, image_path):
    #system_msg: input system message for the LLM
    #prompt_txt: input user prompt for the LLM
    #image_path: the file path of the input image

    ### Credentials of the model
    ### Use a vision-capable model (gemma3 and llama3.2-vision both accept images)
    model = "gemma3"

    ### Step 2.2: Encode the input image to a base64 string
    base64_image = ""
    with open(image_path, "rb") as image_file:
        base64_image = base64.b64encode(image_file.read()).decode("utf-8")

    ### Step 2.3: Define the messages for the model
    ### The image goes in the 'images' field of the user message, not in 'content'
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": prompt_txt, "images": [base64_image]},
    ]

    ### Step 2.4: Get the response for the messages
    model_response = ollama.chat(model=model, messages=messages)

    ### Step 2.5: Return the response (ollama.chat returns the text under message -> content)
    return model_response['message']['content']

In [ ]:
### Define the food image caption prompts given a food name.
### The food name, as you have noticed, comes from the corresponding recipe data.
### You want to include the food name to ensure the model focuses on it while giving captions.
def image_caption_prompt_template(food_name):
    # food_name: the food name of the recipe

    ### Step 3.1: Design the prompts
    image_caption_system_msg = "You are a helpful assistant that describes food images in detail. You will be given an image of a food item and its name. Your task is to provide a detailed description of the food, including its appearance, ingredients, and any notable features."
    image_caption_prompt_txt = f"Describe the food '{food_name}' in the image in detail, including its appearance, ingredients, and any notable features."

    return image_caption_system_msg, image_caption_prompt_txt


### Test the prompts on the first recipe
### Step 3.2: Get the prompts with the food name of the first recipe
image_caption_system_msg, image_caption_prompt_txt = image_caption_prompt_template(recipe1['name'])

### Step 3.3: Get the test response and print it
### The recipe JSON has no image field: images are named after the recipe id (recipe1.png, recipe2.png, ...)
image_path = os.path.join("synthetic_recipe_images", f"recipe{recipe1['id']}.png")
response = vision_llm(image_caption_system_msg, image_caption_prompt_txt, image_path)
print(response)

In [ ]:
### Get captions for each image in the dataset and add to the JSON file (up tp 15 minutes)
### recipe_data is the recipe you loaded in Step 1
for i in range(len(recipe_data)):
    if (i+1)%20 == 0:
        print(f'{i+1} out of {len(recipe_data)} is done')

    ### Step 4.1: Get the caption prompts
    image_caption_system_msg, image_caption_prompt_txt = image_caption_prompt_template(recipe_data[i]['name'])

    image_path = os.path.join("synthetic_recipe_images", f"recipe{recipe_data[i]['id']}.png")
    ### Step 4.2: Get the response with the prompts
    response = vision_llm(image_caption_system_msg, image_caption_prompt_txt, image_path)

    ### Save the response as another item in the recipe data
    recipe_data[i]['image_description'] = response
print('ALL DONE!')

In [ ]:
filename = 'augmented_food_recipe.json'
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(recipe_data, f, indent=4)

In [ ]:
### Part 1

### Step 1.1: Load the review dataset with variable name user_review_data
user_review_data = json.load(open('Synthetic-User-Reviews.json', encoding='utf-8'))

### Step 1.2: Print the first review by key-value pairs

print("First Review:")
for key, value in user_review_data[0].items():
    print(f"  {key}: {value}")

In [ ]:
### Part 2

### Get the image by requesting from the URL
import ast # needed to convert string representation of list to actual python list
import requests

### Convert the list of images of the first user into the actual python list
### Step 1.3: Use ast.literal_eval to convert
images_list = ast.literal_eval(user_review_data[0]['images'])

### Step 1.4: Use the requests.get() method to get the image content
image_content = requests.get(images_list[0]).content


### Step 1.5: Write the image content in Step 1.4 to a temporary file 'review_image_placeholder.jpg'
review_image_path = 'review_image_placeholder.jpg'
with open(review_image_path, 'wb') as f:
    f.write(image_content)

### Step 1.6: Open the 'review_image_placeholder.jpg', and show the image
from PIL import Image
Image.open(review_image_path)

In [ ]:
### Prompt template: caption the images with the context of the reviews
def review_context_image_caption_prompt_template(reviews):
    # reviews: the written review content


    ### Step 2.1: Design your prompts
    review_context_image_caption_system_msg = "You are a culinary expert and a helpful assistant that describes food images in detail. You will be given an image of a food item and its corresponding user reviews. Your task is to provide a detailed description of the food, including its appearance, ingredients, and any notable features, while considering the context provided by the user reviews."
    review_context_image_caption_prompt_txt = f"Based on the following user reviews, provide a detailed description of the food item in the image, including its appearance, ingredients, and any notable features:\n\n{reviews}"

    return review_context_image_caption_system_msg, review_context_image_caption_prompt_txt 

### Step 2.2: Get the prompts
### The review fields are 'title' and 'text' (there is no 'review' key)
reviews = f"{user_review_data[0]['title']}. {user_review_data[0]['text']}"
review_context_image_caption_system_msg, review_context_image_caption_prompt_txt = review_context_image_caption_prompt_template(reviews)

### Step 2.3: Get the response by the vision_llm you defined previously
response = vision_llm(review_context_image_caption_system_msg, review_context_image_caption_prompt_txt, review_image_path)
print(response)

In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential

### URL Request function with Retry
# Retries up to 10 times, starting at 1s and doubling (1s, 2s, 4s...)
@retry(stop=stop_after_attempt(10), wait=wait_exponential(multiplier=1, min=1, max=10))
def get_data_with_retry(url):
    response = requests.get(url, timeout=5)
    response.raise_for_status() # Must raise error for retry to trigger
    return response

### Start the for loop
for i in range(len(user_review_data)):
    ### Step 3.1: Convert the string to the Python list of image urls
    ### ast.literal_eval handles empty lists ('[]') and URLs separated by ',' without a space
    review_images = ast.literal_eval(user_review_data[i]['images'])

    review_image_captions = []
    if len(review_images) > 0:
        for img_url in review_images:
            try:
                ### Step 3.2: Use get_data_with_retry to get the image_data
                image_data = get_data_with_retry(img_url)
                print("Success!")
            except Exception as e:
                print(f"All retries failed at url {img_url}:", e)
                continue
            image = image_data.content
            with open('review_image_placeholder.jpg', 'wb') as img_file:
                img_file.write(image)

            ### Step 3.3: Get the prompts, get the response, and finally append the response to review_image_captions
            reviews = f"{user_review_data[i]['title']}. {user_review_data[i]['text']}"
            review_context_image_caption_system_msg, review_context_image_caption_prompt_txt = review_context_image_caption_prompt_template(reviews)
            response = vision_llm(review_context_image_caption_system_msg, review_context_image_caption_prompt_txt, 'review_image_placeholder.jpg')
            review_image_captions.append(response)
            
    ### Append the review_image_captions to the review data
    user_review_data[i]['image_captions'] = review_image_captions
print('ALL DONE!')

### Save the image-caption-augmented user review data

In [ ]:
filename = 'augmented_user_review.json'
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(user_review_data, f, indent=4)